# AIMENG：Google Colab 免费 GPU 训练入口

本 notebook 面向免费 Colab GPU。它先检查 GPU、磁盘与内存，再获取固定的 Ornith GGUF 并生成 manifest。**GGUF 本身作为冻结推理基线，不进行假装可行的常规反向传播。**

真正训练路由器需要来自实际后端的测量轨迹：每个样本至少包含可观测特征、候选通道组/候选池、质量分数、延迟和峰值内存。没有真实轨迹时，只运行环境检查，不会伪造训练数据或宣称训练成功。

In [ ]:
import os, sys, shutil, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('RAM (GiB):', round(__import__('psutil').virtual_memory().total / 2**30, 2))
print('Free disk (GiB):', round(shutil.disk_usage('/content').free / 2**30, 2))
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except Exception as e:
    print('PyTorch/GPU probe failed:', repr(e))
assert shutil.disk_usage('/content').free > 8 * 2**30, 'Colab 可用磁盘不足 8 GiB；不要开始下载。'


## GPU 显存与吞吐量策略
目标是 16 GiB GPU 上约 14 GiB 的总显存占用，但采用自适应策略，不硬编码分配 14 GiB 张量。优先增加有效 batch / 梯度累积、适度增加序列长度，并在训练真实路由器时使用 BF16（硬件支持时）或 FP16、混合精度和高效 DataLoader。若 GPU 小于 16 GiB，则自动缩小 batch/序列长度。模型下载和 GGUF 推理占用不等于训练路由器的训练显存。

In [ ]:
import torch, json
def gpu_memory_plan(target_gib=14.0, reserve_gib=2.0):
    if not torch.cuda.is_available():
        print('没有检测到 CUDA GPU；先停止，不会伪装成 GPU 训练。')
        return None
    p = torch.cuda.get_device_properties(0)
    total = p.total_memory / 2**30
    free, _ = torch.cuda.mem_get_info(0)
    free_gib = free / 2**30
    target = min(target_gib, max(0.0, total - reserve_gib))
    print(json.dumps({'gpu': p.name, 'total_vram_gib': round(total, 2), 'currently_free_gib': round(free_gib, 2), 'target_total_usage_gib': round(target, 2), 'reserve_gib': round(min(reserve_gib, total), 2), 'note': 'planning ceiling only; never allocate dummy tensors'}, ensure_ascii=False, indent=2))
    if total >= 15.5:
        print('Profile: 16-GiB class; tune real batch/sequence length using measured throughput.')
    else:
        print('Profile: smaller GPU; use conservative settings and do not force 14 GiB allocation.')
    return {'total_gib': total, 'free_gib': free_gib, 'target_gib': target}
GPU_PLAN = gpu_memory_plan()


### 性能验收原则
- 不使用占位张量把显存人为塞满；这只会挤掉真实训练需要的显存，降低吞吐并增加 OOM 风险。
- 训练正式路由器时先测 batch size 1/2/4/8 的 step time 与 tokens/s，选择最快且留有安全余量的配置；如果支持 BF16，优先测试 BF16。
- 以有效训练样本/秒、GPU 利用率和稳定性判断，而不是只看显存占用数字。
- Colab 会话可能被回收；按固定间隔保存路由器 checkpoint 与训练状态到 Google Drive，避免会话结束后从头来过。
- 当前 notebook 的主要工作仍是环境检查和 GGUF 元数据核验；没有真实后端轨迹时，不会启动无意义的路由训练。

## 1. 获取仓库代码
在 Colab 菜单选择 **运行时 → 更改运行时类型 → T4 GPU（如果账号当时提供）**。免费 GPU 的可用性、型号和时长由 Google 动态分配，并不保证每次都有。

In [ ]:
import os, subprocess
REPO = 'https://github.com/bslsjdk/aimeng.git'
BRANCH = 'feat/gguf-router-training-foundation'
if not os.path.isdir('/content/aimeng/.git'):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,'/content/aimeng'], check=True)
else:
    subprocess.run(['git','-C','/content/aimeng','fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C','/content/aimeng','checkout',BRANCH], check=True)
%cd /content/aimeng
!python -m pip install -q -r requirements.txt psutil

## 2. 下载并核对 GGUF
文件约 5.8 GB。Colab 免费实例的磁盘空间会变化；若空间不足，停止而不是反复下载。模型不会上传到 Git。

In [ ]:
from pathlib import Path
free_gib = shutil.disk_usage('/content').free / 2**30
print(f'当前剩余磁盘：{free_gib:.2f} GiB')
if free_gib < 8:
    raise RuntimeError('磁盘空间不足 8 GiB，已停止，避免下载中断。')
subprocess.run([sys.executable, 'scripts/fetch_model.py'], check=True)
MODEL_PATH = Path('models/ornith-1.5-9b-q4km/Ornith-1.5-9B-Q4_K_M.gguf')
subprocess.run([sys.executable, 'scripts/inspect_gguf.py', '--model', str(MODEL_PATH), '--output', 'runs/gguf_manifest.json'], check=True)
print(Path('runs/gguf_manifest.json').read_text()[:6000])

## 3. 可选：准备 llama.cpp 推理后端
训练控制器之前必须先能真实运行 GGUF。下面的构建单元默认关闭，避免点“全部运行”时意外花很久编译；先确认 GPU/磁盘和模型核验成功，再手动把 `BUILD_LLAMA_CPP` 改成 `True` 运行该单元。若构建失败，先解决编译/模型兼容问题，不要继续采集。

In [ ]:
from pathlib import Path
BUILD_LLAMA_CPP = False  # 需要时手动改为 True；CUDA 构建可能耗时较长
LLAMA_CUDA_ENABLED = False
LLAMA_DIR = Path('/content/llama.cpp')
if BUILD_LLAMA_CPP:
    if not (LLAMA_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_DIR)], check=True)
    subprocess.run(['cmake', '-B', 'build', '-DGGML_CUDA=ON'], cwd=LLAMA_DIR, check=True)
    subprocess.run(['cmake', '--build', 'build', '--config', 'Release', '-j', '2', '--target', 'llama-cli'], cwd=LLAMA_DIR, check=True)
    LLAMA_CLI = str(LLAMA_DIR / 'build' / 'bin' / 'llama-cli')
    LLAMA_CUDA_ENABLED = bool(torch.cuda.is_available())
    print('llama-cli:', LLAMA_CLI)
else:
    import shutil as _shutil
    LLAMA_CLI = _shutil.which('llama-cli')
    print('发现的 llama-cli:', LLAMA_CLI or '未安装。需要手动构建或提供兼容二进制。')


## 4. 采集真实预算轨迹（默认关闭）
先准备自己的 `data/tasks.jsonl`，字段格式参考 `examples/tasks.example.jsonl`。正式任务应包含 `dataset_split`、独立验证器和 expected 答案；不要把示例任务当成正式训练集。把 `COLLECTION_MODE` 设为 `smoke` 可用一个示例任务测试 CLI（3 次推理）；设为 `full` 会运行你自己的所有任务，每个任务跑 small/medium/full 三档。完整采集前必须确保任务集已分配足够的 train/validation 任务。

In [ ]:
from pathlib import Path
import json, shutil
COLLECTION_MODE = 'off'  # 'off' / 'smoke' / 'full'
MODEL_PATH = Path('models/ornith-1.5-9b-q4km/Ornith-1.5-9B-Q4_K_M.gguf')
TASK_FILE = Path('data/tasks.jsonl')
PERSIST_TO_DRIVE = False  # 长时间正式采集时改为 True，需授权 Google Drive
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST_ROOT = Path('/content/drive/MyDrive/aimeng')
    PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
    TELEMETRY_PATH = PERSIST_ROOT / 'telemetry.jsonl'
    CONTROLLER_OUTPUT_DIR = PERSIST_ROOT / 'budget_controller'
else:
    TELEMETRY_PATH = Path('data/telemetry.jsonl')
    CONTROLLER_OUTPUT_DIR = Path('outputs/budget_controller')
if COLLECTION_MODE == 'off':
    print('采集已关闭。先检查后端、准备自己的任务文件，再手动选择 smoke 或 full。')
elif not LLAMA_CLI:
    print('STOP: 未找到 llama-cli。先运行上一个可选构建单元或提供兼容二进制。')
elif not MODEL_PATH.is_file():
    raise FileNotFoundError(f'GGUF not found: {MODEL_PATH}')
elif COLLECTION_MODE == 'smoke':
    subprocess.run([sys.executable, 'scripts/collect_budget_traces.py', '--model', str(MODEL_PATH), '--tasks', 'examples/tasks.example.jsonl', '--output', 'runs/smoke_telemetry.jsonl', '--llama-cli', LLAMA_CLI, '--max-tasks', '1', '--gpu-layers', '99' if LLAMA_CUDA_ENABLED else '0', '--accelerator', 'CUDA-T4' if LLAMA_CUDA_ENABLED else 'unknown'], check=True)\n    subprocess.run([sys.executable, 'scripts/validate_telemetry.py', 'runs/smoke_telemetry.jsonl', '--require-paired-budgets', '--check-task-split-leakage'], check=True)
elif COLLECTION_MODE == 'full':
    if not TASK_FILE.is_file():
        raise FileNotFoundError('先创建 data/tasks.jsonl，格式参考 examples/tasks.example.jsonl；示例数据不够训练。')
    tasks = [json.loads(line) for line in TASK_FILE.read_text(encoding='utf-8').splitlines() if line.strip()]
    train_count = sum(1 for task in tasks if task.get('dataset_split') == 'train')
    val_count = sum(1 for task in tasks if task.get('dataset_split') == 'validation')
    test_count = sum(1 for task in tasks if task.get('dataset_split') == 'test')
    if train_count < 12 or val_count < 3 or test_count < 3:
        raise ValueError(f'任务数量不足：train={train_count}（至少 12），validation={val_count}（至少 3），test={test_count}（至少 3）。')
    collect_cmd = [sys.executable, 'scripts/collect_budget_traces.py', '--model', str(MODEL_PATH), '--tasks', str(TASK_FILE), '--output', str(TELEMETRY_PATH), '--llama-cli', LLAMA_CLI, '--gpu-layers', '99' if LLAMA_CUDA_ENABLED else '0', '--accelerator', 'CUDA-T4' if LLAMA_CUDA_ENABLED else 'unknown']
    if TELEMETRY_PATH.is_file():
        collect_cmd.append('--resume')
    subprocess.run(collect_cmd, check=True)
else:
    raise ValueError('COLLECTION_MODE 只能是 off / smoke / full')


In [ ]:
from pathlib import Path
trace = TELEMETRY_PATH
if not trace.exists():
    print(f'STOP: 尚无遥测文件 {trace}。环境与模型准备可以完成，但没有数据就不能训练。')
else:
    subprocess.run([sys.executable, 'scripts/validate_telemetry.py', str(trace), '--require-paired-budgets', '--check-task-split-leakage'], check=True)
    rows = [json.loads(line) for line in trace.read_text(encoding='utf-8').splitlines() if line.strip()]
    train_ids = {r.get('task_id') for r in rows if r.get('task', {}).get('dataset_split') == 'train'}
    val_ids = {r.get('task_id') for r in rows if r.get('task', {}).get('dataset_split') == 'validation'}
    test_ids = {r.get('task_id') for r in rows if r.get('task', {}).get('dataset_split') == 'test'}
    if len(train_ids) < 12 or len(val_ids) < 3 or len(test_ids) < 3:
        print(f'数据格式通过，但暂不训练：train={len(train_ids)}（至少 12），validation={len(val_ids)}（至少 3），test={len(test_ids)}（至少 3）。')
    else:
        subprocess.run([sys.executable, 'scripts/train_budget_controller.py', '--input', str(trace), '--output-dir', str(CONTROLLER_OUTPUT_DIR)], check=True)
        subprocess.run([sys.executable, 'scripts/evaluate_budget_controller.py', '--input', str(trace), '--model-dir', str(CONTROLLER_OUTPUT_DIR), '--output', str(CONTROLLER_OUTPUT_DIR / 'test_report.json')], check=True)
        print('训练和留出测试报告已生成。仍需目标设备通过质量、速度、内存与回退门槛，才可考虑部署。')


## 6. 本次运行的验收标准
- 确认 CUDA GPU 是否真实可用；模型 manifest 包含文件 SHA-256。
- 后端构建/执行失败时停止，不得生成伪造轨迹。
- 每个 task_id 应有且只有一条 small、medium、full 记录；unknown 不会成为训练标签。
- 只有在数据验证通过且 train/validation 数量达标时才运行控制器训练。
- `outputs/budget_controller/manifest.json` 和 `budget_controller.pt` 生成仅表示训练脚本完成，不代表推理加速已证明。
- Android 整应用峰值 RAM 必须独立验证并始终低于 4096 MiB。


## 打开后怎么操作
1. 选择 **运行时 → 更改运行时类型 → T4 GPU**（免费额度提供时）。
2. 选择 **运行时 → 全部运行**，先完成环境检查、依赖安装、GGUF 下载和 manifest 生成。
3. 检查 `llama-cli`：如果未安装，手动将 `BUILD_LLAMA_CPP = True` 后运行可选构建单元。CUDA 构建可能耗时；出现编译错误就先停止并处理错误。
4. 默认 `COLLECTION_MODE = 'off'`，不会擅自启动耗时推理。先用 `smoke` 测试单个示例任务，再准备自己的 `data/tasks.jsonl`。
5. 长时间正式采集时可将 `PERSIST_TO_DRIVE = True`，授权挂载 Google Drive，让遥测与控制器输出持久保存。正式采集前将 `COLLECTION_MODE = 'full'`，任务集至少有 12 个 train、3 个 validation 和 3 个 test 任务，每个任务要有独立可用的 verifier/expected。全部任务每个要跑三档预算；中断后再次运行会尝试跳过已完成的 task/budget。
6. 运行数据校验、训练与独立 test 评估单元。数据不足会安全跳过训练；这不是错误，也不能用随机数据绕过。
7. `PERSIST_TO_DRIVE = True` 时，遥测和最终控制器输出会保存到 Google Drive；关闭时，Colab 会话回收可能导致文件丢失。

**注意：** 当前训练器是独立的小型预算控制器，不会更新 9B GGUF 主模型权重，也不会自动实现 FFN 通道跳算。真实加速必须由后端测量证明。